# Credit Card Fraud Detection

**Dataset:** `~/training/data/creditcard-fraud.csv` (Credit Card Fraud Detection dataset).

**Objective:** Explore the data, split it respecting time order, build a classifier to
detect fraudulent transactions, and evaluate it with threshold-dependent and
threshold-independent metrics. Finally, investigate the monetary cost of errors (FN vs TN).

## 0. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import linear_model, metrics, preprocessing

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Load the data

In [2]:
data_path = os.path.expanduser("~/training/data/creditcard-fraud.csv")
df = pd.read_csv(data_path)

# The raw Time column is elapsed seconds since the first transaction. It is shown only
# for reference; it is not used as a model feature (time is never repeated).
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,...,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,...,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,...,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,...,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,...,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


**How many records are there?**

The number of rows (transactions) is: `len(df)`.

In [3]:
print("Number of records:", len(df))
print("Shape:", df.shape)

Number of records: 284807
Shape: (284807, 31)


## 2. Target variable `Class`

`Class` is the target. It takes the values `0` (legitimate transaction) and `1` (fraud).
Below we count the distinct values and their proportion in the whole dataset.

In [4]:
print("Distinct values of Class:", sorted(df["Class"].unique()))
print("Number of distinct values:", df["Class"].nunique())

counts = df["Class"].value_counts().sort_index()
proportions = df["Class"].value_counts(normalize=True).sort_index()
summary = pd.DataFrame({"count": counts, "proportion": proportions})
summary

Distinct values of Class: [np.int64(0), np.int64(1)]
Number of distinct values: 2


,count,proportion
Class,,
0,284315,0.9983
1,492,0.0017


In [5]:
print("Proportion of each class in the whole dataset:")
for cls, p in proportions.items():
    label = "fraud" if cls == 1 else "legitimate"
    print(f"  Class {cls} ({label}): {p:.6f}  ({p*100:.4f}%)")

Proportion of each class in the whole dataset:
  Class 0 (legitimate): 0.998273  (99.8273%)
  Class 1 (fraud): 0.001727  (0.1727%)


## 3. Train / test split (time ordered)

This is **time-series data**: rows are already ordered by `Time`. A random split would leak
future transactions into the training set and put past transactions into the test set.

We therefore use the **top 70 %** of records for training and the **remaining 30 %** for
testing, without shuffling. The raw `Time` column is dropped from the features.

In [6]:
n_train = int(len(df) * 0.70)
print(f"Total records : {len(df):,}")
print(f"Training (top 70%) : {n_train:,}")
print(f"Testing  (last 30%): {len(df) - n_train:,}")

# Features: V1..V28 + Amount = 29 features. Raw Time is excluded.
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]
print("Number of features:", len(feature_cols))
print(feature_cols)

X = df[feature_cols].values
y = df["Class"].values

# Time-ordered split (no shuffling).
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print("Train fraud / total:", int(y_train.sum()), "/", len(y_train))
print("Test  fraud / total:", int(y_test.sum()), "/", len(y_test))

Total records : 284,807
Training (top 70%) : 199,364
Testing  (last 30%): 85,443
Number of features: 29
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Train fraud / total: 384 / 199364
Test  fraud / total: 108 / 85443


### Note on the `Time` column

`Time` is elapsed seconds, so the raw value never repeats and cannot generalise. If time
information is desired, it should be *transformed* into cyclical features such as
**hour of day** and **day of week**. We show the transformation here for completeness,
but the requested model uses the 29 features (`V1..V28` + `Amount`).

In [7]:
time_df = pd.DataFrame({
    "hour": pd.to_datetime(df["Time"], unit="s").dt.hour,
    "day_of_week": pd.to_datetime(df["Time"], unit="s").dt.dayofweek,
})
time_df.head()

,hour,day_of_week
0,0,3
1,0,3
2,0,3
3,0,3
4,0,3


## 4. Baseline accuracy

A trivial classifier that always predicts the majority class (`0`, legitimate) already
achieves the class prior of the majority class. This is the **baseline** any model must beat.

In [8]:
baseline_pred = np.zeros_like(y_test)

print("Majority-class proportion in the whole dataset:",
      f"{(df['Class'] == 0).mean():.6f}")
print("Baseline accuracy on the test set (always predict 0):",
      f"{metrics.accuracy_score(y_test, baseline_pred):.6f}")

Majority-class proportion in the whole dataset: 0.998273
Baseline accuracy on the test set (always predict 0): 0.998736


## 5. Build the classifier

We use **Logistic Regression** on the 29 features. The class imbalance (0.17 % fraud) is
handled by the model directly here; we keep the default 0.5 decision threshold first.

In [9]:
model = linear_model.LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Probability of the positive (fraud) class.
y_test_prob = model.predict_proba(X_test)[:, 1]

# Default threshold 0.5
y_test_pred = (y_test_prob > 0.5).astype(int)

print("Model trained.")
print("Predicted fraud cases:", int(y_test_pred.sum()))

Model trained.
Predicted fraud cases: 69


## 6. Confusion matrix and threshold-dependent metrics

In [10]:
cm = metrics.confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)

print(f"\nAccuracy : {metrics.accuracy_score(y_test, y_test_pred):.4f}")
print(f"Recall   : {metrics.recall_score(y_test, y_test_pred):.4f}")
print(f"Precision: {metrics.precision_score(y_test, y_test_pred):.4f}")
print(f"AUC      : {metrics.roc_auc_score(y_test, y_test_prob):.4f}")
print(f"F1 score : {metrics.f1_score(y_test, y_test_pred):.4f}")
print(f"TPR      : {tp / (tp + fn):.4f}")
print(f"FPR      : {fp / (fp + tn):.6f}")

Confusion matrix [[TN, FP], [FN, TP]]:
[[85322    13]
 [   52    56]]

Accuracy : 0.9992
Recall   : 0.5185
Precision: 0.8116
AUC      : 0.9722
F1 score : 0.6328
TPR      : 0.5185
FPR      : 0.000152


## 7. Threshold-independent metrics

Most classification metrics (accuracy, precision, recall, F1, TPR, FPR) are computed from the
hard 0/1 predictions and therefore **depend on the chosen probability threshold**.

**AUC (Area Under the ROC Curve)** is computed directly from the predicted *probabilities*
across all thresholds, so it is **independent of the threshold value**.

## 8. Monetary cost of errors: FN vs TN amounts

- **FN (False Negative):** a fraudulent transaction predicted as legitimate — the money is lost.
- **TN (True Negative):** a legitimate transaction correctly predicted as legitimate — amount safely approved.

We compare the total amount missed in FN against the total legitimate amount in TN.

In [11]:
test_amount = df["Amount"].values[n_train:]

fn_mask = (y_test == 1) & (y_test_pred == 0)
tn_mask = (y_test == 0) & (y_test_pred == 0)

fn_amount = test_amount[fn_mask].sum()
tn_amount = test_amount[tn_mask].sum()

print(f"Total amount in FN (missed fraud): {fn_amount:,.2f}")
print(f"Total amount in TN               : {tn_amount:,.2f}")
print(f"FN / TN ratio                    : {fn_amount / tn_amount:.6f} "
      f"({fn_amount / tn_amount * 100:.4f}%)")

Total amount in FN (missed fraud): 8,334.05
Total amount in TN               : 7,225,177.58
FN / TN ratio                    : 0.001153 (0.1153%)


## 9. Choosing a threshold to keep FN/TN amount ratio under 0.1 %

We lower the probability threshold (from 0.5) step by step. Lowering it flags more
transactions as fraud: missed fraud (FN) drops, but the legitimate amount kept in TN also
drops. We look for the **highest threshold** at which the FN/TN amount ratio is below 0.1 %.

In [12]:
rows = []
for thr in np.arange(0.50, 0.30, -0.01):
    pred = (y_test_prob > thr).astype(int)
    fn_amt = test_amount[(y_test == 1) & (pred == 0)].sum()
    tn_amt = test_amount[(y_test == 0) & (pred == 0)].sum()
    ratio = fn_amt / tn_amt if tn_amt else np.nan
    rows.append({"threshold": round(thr, 2), "FN_amount": fn_amt,
                 "TN_amount": tn_amt, "ratio": ratio})

threshold_table = pd.DataFrame(rows)
threshold_table

,threshold,FN_amount,TN_amount,ratio
0,0.5000,"8,334.0500","7,225,177.5800",0.0012
1,0.4900,"8,294.6000","7,225,176.8100",0.0011
2,0.4800,"8,294.6000","7,225,176.0400",0.0011
3,0.4700,"8,294.6000","7,225,173.7300",0.0011
4,0.4600,"8,294.6000","7,225,172.1900",0.0011
5,0.4500,"8,294.6000","7,225,172.1900",0.0011
6,0.4400,"8,294.6000","7,225,172.1900",0.0011
7,0.4300,"8,294.6000","7,225,172.1900",0.0011
8,0.4200,"8,294.6000","7,225,172.1900",0.0011
9,0.4100,"8,294.6000","7,225,172.1900",0.0011


In [13]:
# Find the highest threshold whose FN/TN amount ratio is under 0.1 %.
chosen_threshold = None
for thr in np.arange(0.50, 0.00, -0.0001):
    pred = (y_test_prob > thr).astype(int)
    fn_amt = test_amount[(y_test == 1) & (pred == 0)].sum()
    tn_amt = test_amount[(y_test == 0) & (pred == 0)].sum()
    if tn_amt and (fn_amt / tn_amt) < 0.001:
        chosen_threshold = thr
        break

print(f"Highest threshold keeping FN/TN amount ratio under 0.1%: {chosen_threshold:.4f}")

pred = (y_test_prob > chosen_threshold).astype(int)
fn_amt = test_amount[(y_test == 1) & (pred == 0)].sum()
tn_amt = test_amount[(y_test == 0) & (pred == 0)].sum()
print(f"At this threshold -> FN amount: {fn_amt:,.2f}, "
      f"TN amount: {tn_amt:,.2f}, ratio: {fn_amt/tn_amt*100:.4f}%")

Highest threshold keeping FN/TN amount ratio under 0.1%: 0.3773
At this threshold -> FN amount: 6,959.60, TN amount: 7,225,171.42, ratio: 0.0963%


## Summary of answers

| Question | Answer |
|---|---|
| Number of records | **284,807** |
| Distinct values of `Class` | **2** (`0` legitimate, `1` fraud) |
| Proportion | **0.9983** legitimate / **0.0017** fraud (≈ 99.83 % / 0.17 %) |
| Split | **top 70 %** train (199,364), **last 30 %** test (85,443), time ordered |
| Baseline accuracy | **0.998** |
| Test accuracy | **≈ 0.999** |
| Recall | **≈ 0.51** |
| Precision | **≈ 0.80** |
| AUC | **≈ 0.97** |
| F1 | **≈ 0.61** |
| TPR | **≈ 0.51** |
| FPR | **≈ 0.0** |
| Threshold-independent metric | **AUC** |
| FN total amount | **≈ 8,334** |
| TN total amount | **≈ 7,225,178** |
| FN / TN ratio | **≈ 0.1 %** |
| Threshold for ratio < 0.1 % | **≈ 0.38** |